# Direct Oλ correlations with TCAT programs, scores, and atlas cell types
Estimate 65 marginal Bayesian-bootstrap Pearson correlations across 163 infusion products. The primary table uses within-dataset centering in each draw. The global summary is retained for compatibility with notebooks 61 and 63. Clinical outcomes and transition memberships are not inputs.

Each draw uses one joint feature particle per patient per representation, one independent direct Oλ particle per patient, and a shared set of Bayesian-bootstrap patient weights. Dataset-specific correlations from the same draw determine the probability that all eight datasets share direction.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/Hartert/CAR-T-spectral-organization/DATA')
DIRECT = DATA/'olambda_reciprocal_direct_no_asymptote_floor_v1'
ESTIMATES = DIRECT/'patient_olambda_asymptotes.csv'
SPECTRAL = DIRECT/'patient_olambda_particles.csv'
TCAT = DATA/'patient_TCAT_particles'/'all_patient_TCAT_particles_harmonized.npz'
ATLAS = DATA/'Functional_CART_atlas_patient_particles_hard_labels_all8'/'patient_atlas_particles.npz'
OLD_GEOMETRY = DATA/'atlas_TCAT_correlation_geometry_v1'
OUTPUT = DATA/'direct_Olambda_feature_correlations_v1'
DRAWS, SEED = 4000, 0

In [ ]:
keys = ['Dataset','sample_id']
scores = ['TCAT_SCORE_ASA_CLR','TCAT_SCORE_Proliferation_CLR']
cohort = pd.read_csv(ESTIMATES).set_index(keys,verify_integrity=True).sort_index()
assert len(cohort)==163 and cohort.index.get_level_values('Dataset').nunique()==8
assert cohort.olambda_scale.eq('reciprocal').all() and cohort.model.eq('bounded_direct_no_asymptote_floor').all()
assert cohort.n_cells_available.ge(1000).all()
particles=pd.read_csv(SPECTRAL).set_index(keys,verify_integrity=True).sort_index()
assert particles.index.equals(cohort.index)
columns=[f'particle_{i:02d}' for i in range(1,17)]
assert list(particles.columns)==columns
spectral=particles[columns].to_numpy(float)
assert np.isfinite(spectral).all() and ((spectral>0)&(spectral<1)).all()

def load_states(path,atlas):
    with np.load(path,allow_pickle=False) as a:
        index=pd.MultiIndex.from_arrays([a['datasets'].astype(str),a['patient_ids'].astype(str)],names=keys)
        if not index.is_unique or len(cohort.index.difference(index)):
            raise ValueError(f'Patient identity mismatch in {path}: {cohort.index.difference(index).tolist()}')
        rows=index.get_indexer(cohort.index)
        assert (a['n_cells'][rows]>=1000).all()
        names=a['feature_names'].astype(str)
        keep=np.char.startswith(names,'ATLAS_proportion_' if atlas else 'TCAT_GEP_')
        if not atlas:
            assert keep.sum()==52 and set(scores)<=set(names)
            keep |= np.isin(names,scores)
        names=names[keep]
        assert len(set(names))==len(names)==(11 if atlas else 54)
        values=a['values'][rows][:,:,keep].astype(float)
        weights=a['particle_weights'][rows].astype(float)
    assert np.isfinite(values).all() and values.shape[:2]==weights.shape
    assert np.isfinite(weights).all() and (weights>=0).all() and (weights.sum(axis=1)>0).all()
    cumulative=(weights/weights.sum(axis=1,keepdims=True)).cumsum(axis=1)
    cumulative[:,-1]=1.
    return names,values,cumulative

t_names,t_values,t_cdf=load_states(TCAT,False)
a_names,a_values,a_cdf=load_states(ATLAS,True)
names=np.concatenate([t_names,a_names])
datasets=cohort.index.get_level_values('Dataset').to_numpy()
groups=[np.flatnonzero(datasets==d) for d in sorted(set(datasets))]
assert min(map(len,groups))>=2
display(cohort.reset_index().groupby('Dataset').size().rename('patients').to_frame())

# Read only keys from the old cohort; its particle columns contain the previous target.
prior=OLD_GEOMETRY/'analysis_cohort.csv'
if prior.exists():
    prior_keys=pd.read_csv(prior,usecols=keys).set_index(keys,verify_integrity=True).sort_index().index
    if not prior_keys.equals(cohort.index):
        raise ValueError('Original and direct correlation cohorts differ. Review before reusing feature matrices.')
    print('Matched all original correlation cohort identities.')

In [ ]:
def weighted_r(x,y,w):
    xc=x-w@x
    yc=y-w@y
    cov=(w*yc)@xc
    return cov/np.sqrt(np.maximum((w@np.square(xc))*(w@np.square(yc)),1e-14))

def feature_draw(values,cdf,rng):
    k=(rng.random((len(cdf),1))>=cdf).sum(axis=1)
    return values[np.arange(len(cdf)),k]

rng=np.random.default_rng(SEED)
n=len(cohort)
global_r=np.empty((DRAWS,len(names)),dtype=np.float32)
adjusted_r=np.empty_like(global_r)
within=np.empty((DRAWS,len(groups),len(names)),dtype=np.float64)
for i in range(DRAWS):
    x=np.column_stack([feature_draw(t_values,t_cdf,rng),feature_draw(a_values,a_cdf,rng)])
    y=spectral[np.arange(n),rng.integers(spectral.shape[1],size=n)]
    w=rng.dirichlet(np.ones(n))
    global_r[i]=weighted_r(x,y,w)
    for j,g in enumerate(groups):
        wg=w[g]/w[g].sum()
        within[i,j]=weighted_r(x[g],y[g],wg)
        x[g]-=wg@x[g]
        y[g]-=wg@y[g]
    adjusted_r[i]=weighted_r(x,y,w)
    if (i+1)%250==0 or i+1==DRAWS:
        print(f'Correlation draw {i+1:,}/{DRAWS:,}',flush=True)
assert all(np.isfinite(v).all() for v in (global_r,adjusted_r,within))

In [ ]:
p_all_pos=(within>0).all(axis=1).mean(axis=0)
p_all_neg=(within<0).all(axis=1).mean(axis=0)
def summary(draws):
    lo,med,hi=np.quantile(draws,[.025,.5,.975],axis=0)
    pos=(draws>0).mean(axis=0)
    neg=(draws<0).mean(axis=0)
    return pd.DataFrame(dict(feature=names,rho_median=med,rho_q025=lo,rho_q975=hi,
                             P_rho_gt_0=pos,P_rho_lt_0=neg,P_same_direction=np.maximum(pos,neg)))
tables=[]
for geometry,draws in [('dataset_adjusted',adjusted_r),('global',global_r)]:
    frame=summary(draws)
    frame['geometry']=geometry
    frame['predictor_class']=np.where(np.isin(names,scores),'TCAT score',
                                    np.where(np.isin(names,a_names),'Atlas cell type','TCAT GEP'))
    frame['P_all_datasets_gt_0']=p_all_pos
    frame['P_all_datasets_lt_0']=p_all_neg
    frame['P_all_datasets_same_direction']=p_all_pos+p_all_neg
    tables.append(frame)
results=pd.concat(tables,ignore_index=True)
OUTPUT.mkdir(parents=True,exist_ok=True)
for atlas,filename in [(False,'Olambda_TCAT_GEP_and_score_correlations.csv'),(True,'Olambda_Atlas_cell_type_correlations.csv')]:
    results.loc[results.feature.isin(a_names)==atlas].to_csv(OUTPUT/filename,index=False)
primary=results.loc[results.geometry.eq('dataset_adjusted')]
primary.to_csv(OUTPUT/'direct_Olambda_dataset_adjusted_correlations.csv',index=False)
pd.concat([summary(within[:,j]).assign(Dataset=datasets[g[0]],patients=len(g)) for j,g in enumerate(groups)]).to_csv(
    OUTPUT/'direct_Olambda_within_dataset_correlations.csv',index=False)
np.savez_compressed(OUTPUT/'correlation_draws.npz',feature_names=names,datasets=np.array(sorted(set(datasets)),dtype='U'),
                    dataset_adjusted=adjusted_r,global_r=global_r,within_dataset=within)
cohort.reset_index()[keys].to_csv(OUTPUT/'analysis_cohort.csv',index=False)
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
manifest=dict(target='bounded direct reciprocal O_lambda asymptote particles',draws=DRAWS,seed=SEED,
              patients=n,datasets=len(groups),features=len(names),
              input_sha256={str(p):sha(p) for p in [ESTIMATES,SPECTRAL,TCAT,ATLAS]})
(OUTPUT/'correlation_manifest.json').write_text(json.dumps(manifest,indent=2))
display(primary.sort_values('rho_median',ascending=False))
print('Saved',len(primary),'primary correlations to',OUTPUT)